In [1]:
# PricePilot Phase 3 item 5 — cross-encoder baseline: zero-shot + fine-tuned (v2)
# Changes vs v1: 8 epochs, warmup + linear decay, and the epoch is chosen at each
# epoch's own best validation threshold instead of a fixed 0.5.

import glob, json, random
import numpy as np, torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
SEED, MAX_LEN, EPOCHS, BATCH, LR = 20260923, 256, 8, 16, 2e-5
DEV = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

test_path = glob.glob("/kaggle/input/**/phase3-inputs-test.json", recursive=True)[0]
tv_path   = glob.glob("/kaggle/input/**/phase3-inputs-train-val.json", recursive=True)[0]
TEST, TV = json.load(open(test_path, encoding="utf-8")), json.load(open(tv_path, encoding="utf-8"))
assert TEST["frozen_labels_sha256"] == TV["frozen_labels_sha256"]
print("frozen hash :", TEST["frozen_labels_sha256"])

test_rows, tv_rows = TEST["pairs"], TV["pairs"]
assert len(test_rows) == 287 and len(tv_rows) == 672
assert not any(k in r for r in test_rows for k in ("label", "tier", "split")), "TEST leak!"
train = [r for r in tv_rows if r["train_or_val"] == "train" and r["label"] in ("M", "N")]
val   = [r for r in tv_rows if r["train_or_val"] == "val"   and r["label"] in ("M", "N")]
print(f"train {len(train)}  val {len(val)}  test {len(test_rows)}  device {DEV}")
print(f"val positives: {sum(1 for r in val if r['label']=='M')} / {len(val)}")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

@torch.no_grad()
def score_rows(model, rows, bs=64):
    model.eval(); out = {}
    for i in range(0, len(rows), bs):
        chunk = rows[i:i + bs]
        enc = tok([r["text_a"] for r in chunk], [r["text_b"] for r in chunk],
                  truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEV)
        for r, s in zip(chunk, torch.sigmoid(model(**enc).logits[:, 0].float()).cpu().tolist()):
            out[r["pair_id"]] = float(s)
    return out

def prf(scores, rows, t):
    tp = sum(1 for r in rows if scores[r["pair_id"]] >= t and r["label"] == "M")
    fp = sum(1 for r in rows if scores[r["pair_id"]] >= t and r["label"] == "N")
    fn = sum(1 for r in rows if scores[r["pair_id"]] <  t and r["label"] == "M")
    p = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    return (2 * p * rc / (p + rc) if p + rc else 0.0), p, rc

def best_threshold(scores, rows):
    """Sweep validation only. Returns (best_f1, best_t, p, r)."""
    cands = sorted({round(v, 4) for k, v in scores.items()} | {0.5})
    best = (-1.0, 0.5, 0.0, 0.0)
    for t in cands:
        f, p, r = prf(scores, rows, t)
        if f > best[0]: best = (f, t, p, r)
    return best

# ---------- 1. ZERO-SHOT ----------
zs = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEV)
zs_tv, zs_test = score_rows(zs, tv_rows), score_rows(zs, test_rows)
f, t, p, r = best_threshold(zs_tv, val)
print("zero-shot @0.5      : F1 %.3f P %.3f R %.3f" % prf(zs_tv, val, 0.5))
print("zero-shot @best val : F1 %.3f P %.3f R %.3f  (t=%.4f)" % (f, p, r, t))
del zs; torch.cuda.empty_cache()

# ---------- 2. FINE-TUNE ----------
class PairDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    enc = tok([b["text_a"] for b in batch], [b["text_b"] for b in batch],
              truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor([1.0 if b["label"] == "M" else 0.0 for b in batch])
    return enc

ft = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEV)
dl = DataLoader(PairDS(train), batch_size=BATCH, shuffle=True, collate_fn=collate,
                generator=torch.Generator().manual_seed(SEED))
steps = len(dl) * EPOCHS
opt = torch.optim.AdamW(ft.parameters(), lr=LR)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
lossf = torch.nn.BCEWithLogitsLoss()
print(f"total optimisation steps: {steps}")

best = {"f1": -1.0}
for ep in range(1, EPOCHS + 1):
    ft.train(); total = 0.0
    for batch in dl:
        labels = batch.pop("labels").to(DEV)
        batch = {k: v.to(DEV) for k, v in batch.items()}
        loss = lossf(ft(**batch).logits[:, 0].float(), labels)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ft.parameters(), 1.0)
        opt.step(); sched.step(); total += loss.item()
    vs = score_rows(ft, val)
    f, t, p, r = best_threshold(vs, val)
    f05, p05, r05 = prf(vs, val, 0.5)
    print(f"epoch {ep}  loss {total/len(dl):.4f} | @0.5 F1 {f05:.3f} | @best t={t:.4f} F1 {f:.3f} P {p:.3f} R {r:.3f}")
    if f > best["f1"]:
        best = {"f1": f, "t": t, "p": p, "r": r, "epoch": ep,
                "state": {k: v.detach().cpu().clone() for k, v in ft.state_dict().items()}}

print(f"\nbest epoch {best['epoch']}  val F1 {best['f1']:.4f}  P {best['p']:.3f}  R {best['r']:.3f}"
      f"  val-selected threshold {best['t']:.4f}   (epoch AND threshold chosen on VALIDATION only)")
ft.load_state_dict(best["state"]); ft.to(DEV)
ft_tv, ft_test = score_rows(ft, tv_rows), score_rows(ft, test_rows)

# ---------- 3. WRITE ----------
for name, obj in [("preds-zeroshot-trainval.json",  zs_tv), ("preds-zeroshot-test.json",  zs_test),
                  ("preds-finetuned-trainval.json", ft_tv), ("preds-finetuned-test.json", ft_test)]:
    json.dump(obj, open(f"/kaggle/working/{name}", "w"), indent=1)
    print(name, len(obj), "scores")

assert len(zs_test) == len(ft_test) == 287 and len(zs_tv) == len(ft_tv) == 672
assert not (set(zs_test) & set(zs_tv)), "TEST ids leaked into the TRAIN_VAL file"
print("\nDONE. best_epoch=%d best_val_F1=%.4f val_threshold=%.4f model=%s"
      % (best["epoch"], best["f1"], best["t"], MODEL_NAME))

from IPython.display import FileLink, display
for n in ["preds-zeroshot-trainval.json", "preds-zeroshot-test.json",
          "preds-finetuned-trainval.json", "preds-finetuned-test.json"]:
    display(FileLink(n))

frozen hash : 540a4fd6ccfc52525014ac770caadbf544243dccc5a85d3fcb279fd7d052eed4
train 533  val 133  test 287  device cpu
val positives: 46 / 133


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

total optimisation steps: 272
epoch 1  loss 2.3881 | @0.5 F1 0.584 | @best t=0.5469 F1 0.613 P 0.793 R 0.500
epoch 2  loss 0.4808 | @0.5 F1 0.750 | @best t=0.5400 F1 0.780 P 0.722 R 0.848
epoch 3  loss 0.2801 | @0.5 F1 0.682 | @best t=0.9632 F1 0.814 P 0.875 R 0.761
epoch 4  loss 0.2337 | @0.5 F1 0.769 | @best t=0.8743 F1 0.825 P 0.971 R 0.717
epoch 5  loss 0.1737 | @0.5 F1 0.800 | @best t=0.7796 F1 0.825 P 0.971 R 0.717
epoch 6  loss 0.1135 | @0.5 F1 0.769 | @best t=0.8769 F1 0.840 P 0.971 R 0.739
epoch 7  loss 0.1015 | @0.5 F1 0.778 | @best t=0.8933 F1 0.840 P 0.971 R 0.739
epoch 8  loss 0.0898 | @0.5 F1 0.778 | @best t=0.9298 F1 0.840 P 0.971 R 0.739

best epoch 6  val F1 0.8395  P 0.971  R 0.739  val-selected threshold 0.8769   (epoch AND threshold chosen on VALIDATION only)
preds-zeroshot-trainval.json 672 scores
preds-zeroshot-test.json 287 scores
preds-finetuned-trainval.json 672 scores
preds-finetuned-test.json 287 scores

DONE. best_epoch=6 best_val_F1=0.8395 val_threshold=0.8

/kaggle/working/preds-zeroshot-trainval.json

/kaggle/working/preds-zeroshot-test.json

/kaggle/working/preds-finetuned-trainval.json

/kaggle/working/preds-finetuned-test.json

In [2]:
# Cell 2 — SAVE ONLY, appended for the item-8 retrain (docs/phase3-ce-reproduction-rule.md).
# Cell 1 above is unchanged. This cell trains nothing and changes no score.
import os, json, hashlib, shutil, subprocess, sys
import transformers, tokenizers

OUT = "/kaggle/working/ce-ft-best"
os.makedirs(OUT, exist_ok=True)
ft.save_pretrained(OUT)   # best epoch: cell 1 already ran ft.load_state_dict(best["state"])
tok.save_pretrained(OUT)
print("SAVED:", sorted(os.listdir(OUT)))

for n in ["preds-zeroshot-trainval", "preds-zeroshot-test",
          "preds-finetuned-trainval", "preds-finetuned-test"]:
    os.replace(f"/kaggle/working/{n}.json", f"/kaggle/working/{n}-retrain.json")

st = next(os.path.join(OUT, f) for f in ("model.safetensors", "pytorch_model.bin")
          if os.path.exists(os.path.join(OUT, f)))
try:
    from huggingface_hub import model_info
    hf_sha = model_info(MODEL_NAME).sha
except Exception as e:
    hf_sha = f"UNAVAILABLE: {e!r}"
cpu = subprocess.run(["bash", "-c", "lscpu | grep 'Model name'; nproc"],
                     capture_output=True, text=True).stdout

facts = {
    "model_name": MODEL_NAME, "hf_model_sha": hf_sha,
    "best_epoch": best["epoch"], "val_f1": best["f1"], "notebook_threshold": best["t"],
    "seed": SEED, "device": DEV,
    "param_count": sum(p.numel() for p in ft.parameters()),
    "weights_file": os.path.basename(st), "weights_bytes": os.path.getsize(st),
    "weights_sha256": hashlib.sha256(open(st, "rb").read()).hexdigest(),
    "dir_bytes": sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT)),
    "python": sys.version, "torch": torch.__version__,
    "transformers": transformers.__version__, "tokenizers": tokenizers.__version__,
    "numpy": np.__version__, "cpu": cpu,
}
json.dump(facts, open("/kaggle/working/ce-run-facts.json", "w"), indent=1)
shutil.make_archive("/kaggle/working/ce-ft-best", "zip", OUT)
print(json.dumps(facts, indent=1))
print("best_epoch == 6 ?", best["epoch"] == 6)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SAVED: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
{
 "model_name": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
 "hf_model_sha": "1427fd652930e4ba29e8149678df786c240d8825",
 "best_epoch": 6,
 "val_f1": 0.8395061728395062,
 "notebook_threshold": 0.8769,
 "seed": 20260923,
 "device": "cpu",
 "param_count": 117641089,
 "weights_file": "model.safetensors",
 "weights_bytes": 470588468,
 "weights_sha256": "e8843e39f154419de57c18fda3647434b0feebabf019659631fac8d0ee4dc1fc",
 "dir_bytes": 487356063,
 "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
 "torch": "2.10.0+cpu",
 "transformers": "5.0.0",
 "tokenizers": "0.22.2",
 "numpy": "2.0.2",
 "cpu": "Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz\n4\n"
}
best_epoch == 6 ? True
